### **Orbital Debris Database**   
The Orbital Debris database combines the cleaned UCS and SATCAT datasets into a single analytical source. All de-orbited objects have been removed.   

This notebook builds kinetic_master.csv into a SQLite database using Python, Pandas, and sqlite3, which will be used to run project queries and generate visualizations.

In [1]:
import pandas as pd
import sqlite3
import utility as utils

df_master = pd.read_csv('../data/clean/kinetic_master.csv', low_memory=False)

df = df_master.copy()

conn = sqlite3.connect('../data/clean/orbital_debris.db')

In [ ]:
# Now we have to create a new SQLite database and write the dataframe to it.
# We need to separate the dataframe into multiple tables to avoid redundancy and to make it easier to query later on.
# We also want to patch some of the data that is missing or inconsistent, especially in the ownership metadata, 
# to create a clean ownership_operators table that we can join on later.
# We could have patched this in to the kinetic_master.csv directly, but doing it here allows us to keep the raw 
# master data intact and keeps the data transformation logic in one place. I am still considering breaking each table into
# separate notebooks where we patch the data and export a csv.  Then a final notebook to load the cleaned csvs
# into SQLite. For now, for simplicity, well do it all in this notebook.

# strip white space and standardize case for owner_code.
df['owner_code'] = df['owner_code'].astype(str).str.strip().str.upper()

# strip white space for owner and then standardize common variations of SpaceX to a single canonical name.
df['owner'] = df['owner'].astype(str).str.strip()

owner_name_map = {
    'Spacex': 'SpaceX',
    'spacex': 'SpaceX',
    'spaceX': 'SpaceX',
    'Swarm Technologies': 'SpaceX',
    'Space Exploration Technologies Corp.': 'SpaceX'
}

df['owner'] = df['owner'].replace(owner_name_map)

# aggregate ownership metadata by owner_code
# This will help us create an ownership_operators table without duplicates, 
# and we can then join on the satellite table using owner_code as a foreign key.

# coerce flag columns to numeric, filling non-convertable values with 0, then convert them to int. 
# This ensures that we have consistent 0/1 values for the boolean flags,
flag_cols = ['is_commercial', 'is_government', 'is_military', 'is_civil']

for col in flag_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

# group by owner_code puts all rows for the same owner together then we aggregate the ownership metadata 
# by taking the first non-null value for string fields, and max for boolean flags.  This helps preserve 
# the most complete metadata for each owner while eliminating duplicates. 
owner_profile = df.groupby('owner_code', as_index=False).agg({
    'owner': utils.first_non_null,
    'country_operator': utils.first_non_null,
    'users': utils.first_non_null,
    'is_commercial': 'max',
    'is_government': 'max',
    'is_military': 'max',
    'is_civil': 'max',
    'contractor': utils.first_non_null,
    'contractor_country': utils.first_non_null
})

# only mark non-payload rows as Not Applicable
# this mask could be done a bit more simplier, is_payload = df['object_type'] == 'PAYLOAD'
# but i prefer this way to protect from case drift or accidental whitespace in the object_type field
# which would cause the simpler mask to fail.
#.eq('PAYLOAD') is equivalent to == 'PAYLOAD'.

is_payload = df['object_type'].astype(str).str.strip().str.upper().eq('PAYLOAD')

df.loc[~is_payload, 'primary_purpose'] = df.loc[~is_payload, 'primary_purpose'].fillna('Not Applicable')
df.loc[~is_payload, 'un_registry'] = df.loc[~is_payload, 'un_registry'].fillna('Not Applicable')

# keep lifetime_years numeric + nullable (no forced imputation)
df['lifetime_years'] = pd.to_numeric(df['lifetime_years'], errors='coerce')

# keep orbit_type stable
df['orbit_type'] = df['orbit_type'].fillna('Other/Misc')

# launch_id is synthetic: derive from COSPAR prefix YYYY-NNN
df['launch_id'] = df['cospar_id'].astype(str).str.extract(r'^(\d{4}-\d{3})', expand=False)

df['launch_id'] = df['launch_id'].fillna('UNKNOWN')

df_ownership_operators = owner_profile[
    ['owner_code', 'owner', 'country_operator', 'users',
     'is_commercial', 'is_government', 'is_military', 'is_civil',
     'contractor', 'contractor_country']
]

df_launch_events = df[
    ['launch_id', 'launch_date', 'launch_year', 'launch_site']
].drop_duplicates(subset=['launch_id'])


df_satellites = df[
    ['norad_id', 'cospar_id', 'object_name', 'satellite_name', 'official_name',
     'object_type', 'category', 'ops_status', 'data_status', 'in_orbit',
     'owner_code', 'launch_id']
].drop_duplicates(subset=['norad_id'])

df_orbital_data = df[
    ['norad_id', 'orbit_class', 'orbit_type', 'period_minutes', 'perigee_km',
     'apogee_km', 'inclination_degrees', 'eccentricity', 'semi_major_axis_km',
     'launch_mass_kg', 'proxy_mass_kg', 'dry_mass_kg', 'power_watts',
     'proxy_power_watts', 'rcs', 'rcs_class']
].drop_duplicates(subset=['norad_id'])

df_ucs_details = df[
    ['norad_id', 'lifetime_years', 'sat_age_years',
     'primary_purpose', 'detailed_purpose', 'geo_longitude', 'un_registry']
].drop_duplicates(subset=['norad_id'])

df_risk_assessment = df[
    ['norad_id', 'velocity_kms', 'kinetic_joules', 'is_zombie']
].drop_duplicates(subset=['norad_id'])

# Write each dataframe to SQLite using schema-aligned table names
df_satellites.to_sql('satellites', conn, if_exists='replace', index=False)
df_orbital_data.to_sql('orbital_data', conn, if_exists='replace', index=False)
df_ucs_details.to_sql('ucs_details', conn, if_exists='replace', index=False)
df_risk_assessment.to_sql('risk_assessment', conn, if_exists='replace', index=False)
df_ownership_operators.to_sql('ownership_operators', conn, if_exists='replace', index=False)
df_launch_events.to_sql('launch_events', conn, if_exists='replace', index=False)

# Quick sanity checks (row counts + duplicate key audit)
checks = [
    # TABLE_NAME,           KEY_COLUMN
    ('satellites',          'norad_id'),
    ('orbital_data',        'norad_id'),
    ('ucs_details',         'norad_id'),
    ('risk_assessment',     'norad_id'),
    ('ownership_operators', 'owner_code'),
    ('launch_events',       'launch_id')
]

for table_name, key_col in checks:
    row_count = pd.read_sql(
        f"SELECT COUNT(*) AS count FROM {table_name}",
        conn).iloc[0]['count']
    
    dup_count = pd.read_sql(
        f"SELECT COUNT(*) AS count FROM (SELECT {key_col} FROM {table_name} GROUP BY {key_col} HAVING COUNT(*) > 1)",
        conn).iloc[0]['count']
        
    print(f"{table_name:20} | rows = {int(row_count):<5} | duplicate_{key_col:<10} = {int(dup_count):<10}")

conn.commit()
conn.close()

print('\nSQLite build complete: ../data/clean/orbital_debris.db')

Unique launch_id count: 3933 (should be less than total rows: 33258)
satellites           | rows = 33258 | duplicate_norad_id   = 0         
orbital_data         | rows = 33258 | duplicate_norad_id   = 0         
ucs_details          | rows = 33258 | duplicate_norad_id   = 0         
risk_assessment      | rows = 33258 | duplicate_norad_id   = 0         
ownership_operators  | rows = 105   | duplicate_owner_code = 0         
launch_events        | rows = 3933  | duplicate_launch_id  = 0         

SQLite build complete: ../data/clean/orbital_debris.db
